# DTU Chatbot - Retrieval Augmenting Generation on Master’s program specifications and DTU’s courses

## Introduction

Finding information about master’s programs and courses at DTU can be surprisingly difficult for students. Details about study programs are scattered across different parts of the DTU website, and with more than 700 master's courses available, it’s not easy to get a full overview. The current search functionality doesn’t make it much easier — finding and comparing courses often becomes a frustrating and time-consuming task.

### Motivation

Many students struggle to make informed decisions about their studies simply because it’s hard to access the right information quickly. Exploring different study options, checking course prerequisites, and understanding program structures should be straightforward, but today it often isn’t.

To address this, we developed a chatbot that allows users to ask natural language questions and instantly retrieve relevant information about DTU’s programs and courses. The goal is to make the process of finding and understanding study options much simpler, more efficient, and less frustrating. 

The chatbot is deployed and available here: [DTU Chatbot](https://dtu-chatbot.streamlit.app/).

The whole code is available on [GitHub](https://github.com/RasSoender/RAG_DTU).

# Chatbot Architecture Overview

The chatbot is built through a modular pipeline that handles everything from raw data collection to user interaction and response generation. Here's an overview of how it works:

### 1. Data Collection and Preparation
- **Web Scraping:** The raw data is scraped directly from DTU’s official web pages. This includes detailed descriptions of master's programs and individual courses.
- **Preprocessing:** After scraping, documents are cleaned and preprocessed to standardize formatting, remove noise, and organize the content.
- **Embedding Models:** The cleaned data is then encoded using **multiple embedding models**. Different models are used to better capture the semantic differences between course content and study program descriptions.

### 2. Storage and Retrieval
- **Separate Vector Databases:** 
  - One vector database is dedicated to **courses**.
  - Another vector database is used for **master’s programs**.
- **Retrieval:** When a user submits a query, the system searches the appropriate database based on the query type. Retrieval is based on:
  - **Keyword search** (for exact matching).
  - **Cosine similarity** (for semantic matching using the embeddings).

### 3. User Interaction and Query Handling
- **User Interface:** Students interact with the chatbot through a simple web-based interface where they can ask questions and receive answers.
- **Query Routing:** The system classifies the user's query to determine whether it relates to a course or a study program and routes it to the correct vector database.
- **Short-Term Memory:** A short-term memory component ("Brain") keeps track of previous exchanges in the session to maintain context and improve multi-turn conversations.

### 4. Response Generation
- **LLM (Large Language Model):** Retrieved documents are passed to a large language model, which generates a coherent and natural-sounding response to the user’s query.

### 5. Feedback Loop
- **User Feedback:** Students can provide feedback about the relevance and quality of the answers. This feedback is stored in a MongoDB database and can be used for future model fine-tuning and system improvements.

The whole pipeline is illustrated in the following figure:

![Chatbot Architecture](./images/chatbot_schema.jpeg)

## 1. Data Collection and Preparation

### Web Scraping

One of the core components of a Retrieval-Augmented Generation (RAG) is the information its retrieve and base its answers on. The chatbot should be able to answer questions about master's programs and master's courses on DTU, and it was therefore nessecary to gather those information.

No database or data collection of DTU's master's program or courses was publicly accessible to use for the RAG. Therefore webscraping was applied to collect the data nessecary for the project. The two webpages are https://kurser.dtu.dk/ (about courses) and https://sdb.dtu.dk/ (about programmes) contains all the information needed. 

To different libraries was used for webscraping the two webpages: 
- crawl4ai
- selenium

Crawl4ai is an asynchronous web crawler with allows the user to easily scrape urls and convert into markdown format. This was used for the scraping of the program information since all data in each url was needed and no further specification was needed. 

Initially crawl4ai was attempted to use on retrieving the course information, but this was not possible, because (https://kurser.dtu.dk/) needed authorisation/log-in which crawl4ai was not able to provide. Instead we opted to utilize Selenium, because it waits for the webpages to render and therefore was able to overcome the authorisation problem. The downside of this was the more HTML-specification and longer running-times. 


In [ ]:

# -------------------

# The file used for scraping program information with crawl4ai

# -----------------

import asyncio
from crawl4ai import AsyncWebCrawler
from crawl4ai.async_configs import BrowserConfig, CrawlerRunConfig, CacheMode
from typing import List
import json

def save_to_file(title, result):
    folder = "data/data_study_programmes"
    filename = f"{folder}/{title.replace(' ', '_')}.json"
    with open(filename, "w", encoding="utf-8") as f:
        result_dict = {
            "markdown": result.markdown
        }
        json.dump(result_dict, f, ensure_ascii=False, indent=2)
    print(f"Result saved to {filename}")


async def sequntial_crawl(urls, titles):
    browser_config = BrowserConfig(
        headless=True,
        verbose=True,
        extra_args=["--disable-gpu", "--disable-dev-shm-usage", "--no-sandbox"]
    )

    crawl_config = CrawlerRunConfig(cache_mode=CacheMode.BYPASS)

    crawler = AsyncWebCrawler(config=browser_config)
    print("starting Crawler")
    await crawler.start()

    try:
        for i, url in enumerate(urls): 
            print(f"Crawling {url}...")

            result = await crawler.arun(
                url=url,
                config=crawl_config
            )
            if result.success:
                print(result.url, "crawled OK!")
                save_to_file(titles[i], result)
            else:
                print("Failed", result.url,"-", result.error_message)

    except Exception as e: 
        print(f"Error: {e}")
    finally:
        print("Closing crawler")
        await crawler.close()


if __name__ == "__main__":
    with open("reference_data.json", "r", encoding="utf-8") as file:
        data = json.load(file)

    # Extract the arrays
    study_programme_urls = data.get("study_programme_urls", [])
    study_programme_titles = data.get("study_programme_titles", [])
    
    asyncio.run(sequntial_crawl(study_programme_urls, study_programme_titles))

In [ ]:

# -------------------

# Small codesnippet from the course scraping file. 
# Two functions used for scraping specific HTML elements. 

# -----------------

def scraping_elements_left(driver, j, scraped_info):
    # Get all rows at once to reduce DOM queries
    try:
        rows = driver.find_elements(By.XPATH, f"//div[@class='box information']//table[{j}]//tr")
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                key = cols[0].text.strip()
                value = cols[1].text.strip()
                if key:  # Only add if key exists
                    scraped_info[key] = value
    except Exception as e:
        print(f"Error in scraping_elements_left: {e}")
    return scraped_info

def scraping_elements_right(driver, scraped_info):
    try:
        # Get all section titles at once
        all_keys = []
        keys_elements = driver.find_elements(By.XPATH, "//div[@class='col-md-6 col-sm-12 col-xs-12']//div[@class='box']//div[@class='bar']")
        for element in keys_elements:
            key = element.text.strip()
            if key:
                all_keys.append(key)
        
        # Get the full text content
        all_text = scrape_text(driver, "//div[@class='col-md-6 col-sm-12 col-xs-12']//div[@class='box']")
        
        if all_text and all_keys:
            # Split text by titles
            split_text_by_titles(all_text, all_keys, scraped_info)
    except Exception as e:
        print(f"Error in scraping_elements_right: {e}")
    
    return scraped_info

To further enrich the possible answers and details that the chatbot can give, it was decided to also scrape the webpage https://dtucourseanalyzer.pythonanywhere.com/. This webpage contains information about courses which is either based on student inputs or otherwise not available on the offical DTU webpage (i.e. workload burden, avg. rating, failed students, avg. grade, etc.). 

Selenium was utilized, because only specific information was necesserary which could be easilier specified with Selenium. A code snippet is shown below: 

In [ ]:
def scraping_elements(driver, scraped_info):
    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][1]//tr[5]//td"
    signups = scrape_text(driver, xpath)
    scraped_info["signups"] = signups

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][2]//table[2]//tr[1]//td"
    average_grade = scrape_text(driver, xpath)
    scraped_info["average grade"] = average_grade

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][2]//table[2]//tr[2]//td"
    failed_students = scrape_text(driver, xpath)
    scraped_info["failed students in percent"] = failed_students

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][3]//tr[1]//td"
    workload_burden = scrape_text(driver, xpath)
    scraped_info["workload burden"] = workload_burden

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][3]//tr[2]//td"
    overworked_students = scrape_text(driver, xpath)
    scraped_info["overworked students in percent"] = overworked_students

    xpath = f"//div[@class='container']//div[@class='col']//li[@class='list-group-item'][4]//tr[1]//td"
    average_rating = scrape_text(driver, xpath)
    scraped_info["average rating"] = average_rating

    return scraped_info

def process_url(url, index):
    try:
        print(f"Processing URL {index}")
        driver = setup_driver(headless=True)
        
        # Set timeout to avoid hanging on problematic pages
        driver.set_page_load_timeout(30)
        
        try:
            driver.get(url)
        except Exception as e:
            print(f"Error loading URL {url}: {e}")
            driver.quit()
            return False
            
        # Refreshing the page
        driver.refresh()
        time.sleep(1)  # Reduced sleep time
        
        scraped_info = dict()
        scraped_info["course title"] = scrape_text(driver, "//h5")
        
        if not scraped_info["course title"]:
            print(f"Failed to get course title for URL {url}")
            driver.quit()
            return False

        scraped_info = scraping_elements(driver, scraped_info)

        # Save results
        saving_into_json(scraped_info)
        
        driver.quit()
        return True
    except Exception as e:
        print(f"Error processing URL {url}: {e}")
        return False

Even though the webscraping is slow, it only need to be done one time (or possible more if some information are updated). 

A lot of other small details as collecting the urls, cleaning the scraped data, converting to json-format, merging files etc. was also done, but did seem unnecessary and nonessential to include in this technical overview.   